**CI twin of `ch02-linear-regression.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
import matplotlib.pyplot as plt

homes = load_csv("california-housing-sample")
prices = homes["MedHouseVal"]

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.scatter(homes["MedInc"], prices, s=8, alpha=0.4)
ax.set_xlabel("median income ($10k units)")
ax.set_ylabel("median house value ($100k units)")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

X = homes[["MedInc"]]          # features: a table with one column
model = LinearRegression()
model.fit(X, prices)

print(f"learned weight w:    {model.coef_[0]:.3f}")
print(f"learned intercept b: {model.intercept_:.3f}")

In [ ]:
import pandas as pd

new_homes = pd.DataFrame({"MedInc": [3.0, 8.0]})
print(model.predict(new_homes).round(3))
print("by hand:", round(0.432 * 3.0 + 0.410, 3), "and",
      round(0.432 * 8.0 + 0.410, 3))

In [ ]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X)

gaps = [abs(t - p) for t, p in zip(prices, predictions)]
print(f"mae, written by you:  {sum(gaps) / len(gaps):.3f}")
print(f"sklearn's version:    {mean_absolute_error(prices, predictions):.3f}")

In [ ]:
features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]

model8 = LinearRegression()
model8.fit(homes[features], prices)

pred8 = model8.predict(homes[features])
print(f"MAE with 8 features: {mean_absolute_error(prices, pred8):.3f}\n")
for name, w in zip(features, model8.coef_):
    print(f"  {name:11} {w:+.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression

X = [[1.0], [2.0], [3.0], [4.0]]
y = [3.0, 5.0, 7.0, 9.0]
model = LinearRegression()
model.fit(X, y)

run_tests([
    ("slope", round(float(model.coef_[0]), 1), 2.0),
    ("intercept", round(float(model.intercept_), 1), 1.0),
    ("predict 5 hours", round(float(model.predict([[5.0]])[0]), 1), 11.0),
])

In [ ]:
def fit_line(xs, ys):
    xbar = sum(xs) / len(xs)
    ybar = sum(ys) / len(ys)
    num = sum((x - xbar) * (y - ybar) for x, y in zip(xs, ys))
    den = sum((x - xbar) ** 2 for x in xs)
    w = num / den
    return w, ybar - w * xbar

run_tests([
    ("slope (noisy line)",
     fit_line([1.0, 2.0, 3.0, 4.0, 5.0], [2.0, 2.5, 4.0, 4.5, 6.0])[0], 1.0),
    ("intercept (noisy line)",
     fit_line([1.0, 2.0, 3.0, 4.0, 5.0], [2.0, 2.5, 4.0, 4.5, 6.0])[1], 0.8),
    ("slope (perfect line)",
     fit_line([1.0, 2.0, 3.0, 4.0], [3.0, 5.0, 7.0, 9.0])[0], 2.0),
    ("intercept (perfect line)",
     fit_line([1.0, 2.0, 3.0, 4.0], [3.0, 5.0, 7.0, 9.0])[1], 1.0),
], tol=1e-9)